# Dataset Analysis & Antimicrobial Resistance Exploration
### AI-Based Antibiotic Resistance Intelligence System (AMR-IS)
**Dataset:** CDC & FDA National Antimicrobial Resistance Monitoring System (NARMS Now)
**Records:** 54,351 real surveillance isolates (1996–2015)

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cleaned dataset and schema
df = pd.read_csv('../data/processed/narms_cleaned.csv', low_memory=False)
with open('../config/feature_schema.json', 'r') as f:
    schema = json.load(f)

print(f"Cleaned dataset shape: {df.shape}")
df.head()

## 1. Feature Information & Missingness

In [ ]:
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})
info_df

## 2. Organism Distribution (Genus and Species)

In [ ]:
genus_counts = df['Genus'].value_counts()
print("Genus Counts:")
print(genus_counts)

species_counts = df['Species'].value_counts().head(10)
print("\nTop 10 Species Counts:")
print(species_counts)

## 3. Antibiotic Resistance Target Distributions (S vs R)

In [ ]:
target_cols = [c for c in df.columns if c.startswith('target_')]
target_stats = []

for tc in target_cols:
    abx_name = tc.replace('target_', '').capitalize()
    valid = df[tc].dropna()
    s_count = (valid == 0).sum()
    r_count = (valid == 1).sum()
    total = len(valid)
    r_pct = (r_count / total * 100) if total > 0 else 0
    target_stats.append({
        'Antibiotic': abx_name,
        'Susceptible (0)': s_count,
        'Resistant (1)': r_count,
        'Total Tested': total,
        'Resistance Rate (%)': round(r_pct, 2)
    })

pd.DataFrame(target_stats)

## 4. Surveillance Year Trends (1996–2015)

In [ ]:
year_dist = df['Data_Year'].value_counts().sort_index()
print("Isolates per Year:")
print(year_dist)

## 5. Leakage Prevention Verification
- All post-outcome columns, resistance determinants, and intermediate testing results are isolated.
- Only microbiological, temporal, and patient context features (Genus, Species, Serotype_Grouped, Region_Name, Age_Group, Specimen_Source, Data_Year) are retained for modeling.